In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest

print("Ready.")

Ready.


In [2]:
df = pd.read_csv("omantel_network_activity.csv")

print(df.shape)
df.head()

(2500, 9)


,msisdn,daily_calls,avg_call_sec,intl_minutes,unique_numbers,night_call_pct,data_usage_gb,is_fraud,fraud_type
0,96895000000,15.5,249.2,19.0,10.6,3.3,32.5,0,NaN
1,96895000001,12.9,275.8,17.3,8.3,18.8,10.2,0,NaN
2,96895000002,9.8,89.4,3.1,9.3,25.4,5.3,0,NaN
3,96895000003,10.5,121.0,18.2,12.0,24.4,9.8,0,NaN
4,96895000004,8.8,135.6,12.6,12.6,11.0,0.1,0,NaN


In [4]:
FEATURES = [
    "daily_calls",
    "avg_call_sec",
    "intl_minutes",
    "unique_numbers",
    "night_call_pct",
    "data_usage_gb"
]

df[FEATURES].describe().round(1)

,daily_calls,avg_call_sec,intl_minutes,unique_numbers,night_call_pct,data_usage_gb
count,2500.0,2500.0,2500.0,2500.0,2500.0,2500.0
mean,14.6,147.2,21.2,11.0,12.3,18.7
std,20.2,61.0,58.0,15.9,8.1,12.9
min,0.0,5.0,0.0,1.0,0.0,0.1
25%,8.3,105.9,4.9,6.4,6.6,8.2
50%,12.4,147.9,11.8,9.1,11.9,18.0
75%,16.3,188.4,19.2,11.9,17.1,27.6
max,259.4,385.6,514.7,176.7,66.5,66.0


In [5]:
print("Total SIMs on the network:", len(df))

print("Fraudulent SIMs hiding in here:", df["is_fraud"].sum())

print(
    "That is",
    round(df["is_fraud"].mean()*100,2),
    "percent"
)

Total SIMs on the network: 2500
Fraudulent SIMs hiding in here: 40
That is 1.6 percent


In [6]:
rule = df["intl_minutes"] > 300

print("SIMs the rule sends for review :", rule.sum())

print("Of those, actually fraud :", df.loc[rule, "is_fraud"].sum())

print(
    "FALSE ALARMS :",
    (rule & (df["is_fraud"] == 0)).sum()
)

print(
    "Real fraud MISSED :",
    ((~rule) & (df["is_fraud"] == 1)).sum()
)

print("\nWhat it missed, by type:")

print(
    df[
        (~rule) & (df["is_fraud"] == 1)
    ]["fraud_type"].value_counts()
)

SIMs the rule sends for review : 50
Of those, actually fraud : 12
FALSE ALARMS : 38
Real fraud MISSED : 28

What it missed, by type:
fraud_type
Spam-Caller    18
SIM-Box        10
Name: count, dtype: int64


In [7]:
false_alarms = df[
    rule &
    (df["is_fraud"] == 0)
]

print("FALSE ALARMS")

print(
    false_alarms[FEATURES].mean().round(1)
)

missed = df[
    (~rule) &
    (df["is_fraud"] == 1)
]

print("\nMISSED FRAUD")

print(
    missed[FEATURES].mean().round(1)
)

FALSE ALARMS
daily_calls        14.1
avg_call_sec      169.6
intl_minutes      405.8
unique_numbers      7.2
night_call_pct     10.7
data_usage_gb      14.7
dtype: float64

MISSED FRAUD
daily_calls       152.7
avg_call_sec       30.7
intl_minutes       88.4
unique_numbers    131.2
night_call_pct     22.2
data_usage_gb       2.9
dtype: float64


In [8]:
iso = IsolationForest(
    contamination=0.02,
    random_state=42
)

df["flag"] = (
    iso.fit_predict(df[FEATURES]) == -1
).astype(int)

print("SIMs flagged as anomalous:", df["flag"].sum())

SIMs flagged as anomalous: 50


In [9]:
print("=== RULE ===")

print("Sent for review:", rule.sum())
print("Actually fraud:", df.loc[rule, "is_fraud"].sum())
print(
    "False alarms:",
    (rule & (df["is_fraud"]==0)).sum()
)
print(
    "Fraud missed:",
    ((~rule) & (df["is_fraud"]==1)).sum()
)

print("\n=== ISOLATION FOREST ===")

print("Sent for review:", df["flag"].sum())
print(
    "Actually fraud:",
    df.loc[df["flag"]==1,"is_fraud"].sum()
)
print(
    "False alarms:",
    ((df["flag"]==1) & (df["is_fraud"]==0)).sum()
)
print(
    "Fraud missed:",
    ((df["flag"]==0) & (df["is_fraud"]==1)).sum()
)

=== RULE ===
Sent for review: 50
Actually fraud: 12
False alarms: 38
Fraud missed: 28

=== ISOLATION FOREST ===
Sent for review: 50
Actually fraud: 40
False alarms: 10
Fraud missed: 0


In [10]:
df.groupby("flag")[FEATURES].mean().round(1)

,daily_calls,avg_call_sec,intl_minutes,unique_numbers,night_call_pct,data_usage_gb
flag,,,,,,
0,12.3,148.9,16.8,9.1,12.1,19.0
1,130.7,64.0,237.2,105.8,26.4,4.6


In [11]:
queue = df[df["flag"]==1][
    ["msisdn"] + FEATURES +
    ["is_fraud","fraud_type"]
]

queue = queue.sort_values(
    "daily_calls",
    ascending=False
)

queue.to_csv(
    "omantel_fraud_review_queue.csv",
    index=False
)

print("Report generated successfully.")
print("SIMs in queue:", len(queue))

queue.head(10)

Report generated successfully.
SIMs in queue: 50


,msisdn,daily_calls,avg_call_sec,intl_minutes,unique_numbers,night_call_pct,data_usage_gb,is_fraud,fraud_type
1303,96895001303,259.4,45.4,381.5,134.3,49.4,3.2,1,SIM-Box
598,96895000598,251.9,48.2,254.3,139.0,56.8,4.2,1,SIM-Box
478,96895000478,228.5,18.9,185.9,161.5,59.1,2.7,1,SIM-Box
9,96895000009,222.2,13.9,442.1,140.1,58.3,1.6,1,SIM-Box
1372,96895001372,219.9,60.0,341.8,119.1,44.1,4.2,1,SIM-Box
221,96895000221,215.0,55.0,152.5,129.4,41.0,4.8,1,SIM-Box
1146,96895001146,214.3,22.2,406.5,106.9,47.8,5.0,1,SIM-Box
806,96895000806,197.6,38.7,283.2,100.1,41.2,2.3,1,SIM-Box
1652,96895001652,197.4,52.6,260.5,166.7,48.6,4.7,1,SIM-Box
950,96895000950,186.7,39.8,256.0,135.1,53.2,4.8,1,SIM-Box


In [12]:
pd.crosstab(
    df["flag"],
    df["fraud_type"]
)

fraud_type,SIM-Box,Spam-Caller
flag,,
1,22,18
